 Análise de Licitações PNCP
 Pipeline analítico — Tratamento e exploração
 ============================================================
 Seções:
   1. Configuração e Imports
   2. Leitura e Separação dos CSVs
   3. Padronização de Tipos
   4. Filtros e Deduplicação
   5. Merge e Construção da Bigtable
   6. Feature Engineering
   7. Limpeza Final e Salvamento
   8. Análise Exploratória (EDA)

In [1]:
import os
import warnings
from pathlib import Path
 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
 
 
PASTA_DADOS   = Path("./DataBases")
ARQUIVO_BASE  = Path("base_analitica_licitacoes.parquet")
ARQUIVO_FINAL = Path("base_analitica_licitacoes_tratada.parquet")
 
# Colunas que devem ser lidas como string para evitar mixed-types
COLUNAS_STR_COMPRA = [
    "id_compra", "numero_controle_PNCP", "cod_compra",
    "orgao_entidade_cnpj", "orgao_subrogado_cnpj",
    "codigo_orgao", "unidade_orgao_codigo_unidade",
]
COLUNAS_STR_ITEM = [
    "id_compra", "id_compra_item", "srk_pncp_item_compra",
    "cod_compra", "cod_item_compra",
    "unidade_orgao_codigo_unidade",
]
COLUNAS_STR_RESULTADO = [
    "id_compra", "id_compra_item", "srk_item_resultado",
    "ni_fornecedor", "unidade_orgao_codigo_unidade",
]
 
 
print("Configuração carregada.")

Configuração carregada.


2. Leitura e Separação dos CSVs

In [2]:
def ler_csvs(pasta: Path, colunas_str: list[str]) -> pd.DataFrame:
    """
    Lê todos os CSVs de uma pasta que batem com o padrão do nome,
    forçando dtype=str nas colunas problemáticas para evitar DtypeWarning.
    """
    arquivos = list(pasta.glob("*.csv"))
    lista = []
 
    for caminho in arquivos:
        nome = caminho.name
        print(f"  Lendo: {nome}")
        try:
            dtype_map = {col: str for col in colunas_str}
            df_temp = pd.read_csv(
                caminho,
                sep=",",
                encoding="utf-8",
                dtype=dtype_map,
                low_memory=False,
            )
            lista.append(df_temp)
        except Exception as e:
            print(f"Erro ao ler {nome}: {e}")
 
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()
 
 
print("Iniciando leitura e separação dos arquivos...\n")
arquivos = list(PASTA_DADOS.glob("*.csv"))
 
lista_compra    = []
lista_item      = []
lista_resultado = []
 
for caminho in arquivos:
    nome = caminho.name
    print(f"Lendo: {nome}")
    try:
        if "COMPRA_ITEM" in nome:
            dtype_map = {col: str for col in COLUNAS_STR_ITEM}
        elif "ITEM_RESULTADO" in nome:
            dtype_map = {col: str for col in COLUNAS_STR_RESULTADO}
        else:
            dtype_map = {col: str for col in COLUNAS_STR_COMPRA}
 
        df_temp = pd.read_csv(
            caminho, sep=",", encoding="utf-8",
            dtype=dtype_map, low_memory=False,
        )
 
        if "COMPRA_ITEM" in nome:
            lista_item.append(df_temp)
        elif "ITEM_RESULTADO" in nome:
            lista_resultado.append(df_temp)
        elif "COMPRA-" in nome:
            lista_compra.append(df_temp)
 
    except Exception as e:
        print(f"Erro ao ler {nome}: {e}")
 
df_compra_final    = pd.concat(lista_compra,    ignore_index=True) if lista_compra    else pd.DataFrame()
df_item_final      = pd.concat(lista_item,      ignore_index=True) if lista_item      else pd.DataFrame()
df_resultado_final = pd.concat(lista_resultado, ignore_index=True) if lista_resultado else pd.DataFrame()
 
print(f"\n Leitura concluída.")
print(f"   COMPRA    : {df_compra_final.shape}")
print(f"   ITEM      : {df_item_final.shape}")
print(f"   RESULTADO : {df_resultado_final.shape}")

Iniciando leitura e separação dos arquivos...

Lendo: comprasGOV-anual-VW_DM_PNCP_ITEM_RESULTADO-2024.csv
Lendo: comprasGOV-anual-VW_DM_PNCP_ITEM_RESULTADO-2025.csv
Lendo: comprasGOV-anual-VW_FT_PNCP_COMPRA-2024.csv
Lendo: comprasGOV-anual-VW_FT_PNCP_COMPRA-2025.csv
Lendo: comprasGOV-anual-VW_FT_PNCP_COMPRA_ITEM-2024.csv
Lendo: comprasGOV-anual-VW_FT_PNCP_COMPRA_ITEM-2025.csv

 Leitura concluída.
   COMPRA    : (618846, 72)
   ITEM      : (6379194, 62)
   RESULTADO : (4395570, 52)


3. Padronização de Tipos

In [3]:
def padronizar_chaves(df: pd.DataFrame, colunas: list[str]) -> pd.DataFrame:
    """Garante que as colunas-chave de join sejam sempre string."""
    for col in colunas:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    return df
 
# Colunas de join entre as três tabelas
CHAVES_JOIN = ["id_compra", "id_compra_item"]
 
df_compra    = padronizar_chaves(df_compra_final.copy(),    CHAVES_JOIN)
df_item      = padronizar_chaves(df_item_final.copy(),      CHAVES_JOIN)
df_resultado = padronizar_chaves(df_resultado_final.copy(), CHAVES_JOIN)
 
# Converter datas
COLUNAS_DATA_COMPRA = [
    "data_inclusao", "data_atualizacao_pncp",
    "data_publicacao_pncp", "data_abertura_proposta_pncp",
    "data_encerramento_proposta_pncp",
]
COLUNAS_DATA_RESULTADO = [
    "data_inclusao", "data_atualizacao", "data_resultado",
    "data_resultado_pncp", "data_cancelamento",
]
 
for col in COLUNAS_DATA_COMPRA:
    if col in df_compra.columns:
        df_compra[col] = pd.to_datetime(df_compra[col], errors="coerce")
 
for col in COLUNAS_DATA_RESULTADO:
    if col in df_resultado.columns:
        df_resultado[col] = pd.to_datetime(df_resultado[col], errors="coerce")
 
print("Tipos padronizados.")

Tipos padronizados.


4. Filtros e Deduplicação

In [4]:
# ── 4.1 Filtrar resultados válidos ─────────────────────────────────────────
df_resultado = df_resultado[
    df_resultado["situacao_compra_item_resultado_nome"] == "Informado"
].copy()
print(f"Após filtro 'Informado': {df_resultado.shape}")
 
# ── 4.2 Deduplicação ───────────────────────────────────────────────────────
# COMPRA: 1 linha por id_compra (mais recente)
df_compra = (
    df_compra
    .sort_values(["id_compra", "data_atualizacao_pncp"], ascending=[True, False], na_position="last")
    .drop_duplicates(subset=["id_compra"], keep="first")
    .copy()
)
 
# ITEM: 1 linha por (id_compra, id_compra_item)
df_item = (
    df_item
    .sort_values(["id_compra", "id_compra_item"], ascending=[True, True], na_position="last")
    .drop_duplicates(subset=["id_compra", "id_compra_item"], keep="first")
    .copy()
)
 
# RESULTADO: idealmente 1 linha por (id_compra, id_compra_item) — mantém o mais recente
df_resultado = (
    df_resultado
    .sort_values(["id_compra", "id_compra_item", "data_resultado"], ascending=[True, True, False], na_position="last")
    .drop_duplicates(subset=["id_compra", "id_compra_item"], keep="first")
    .copy()
)
 
print(f"\n Após deduplicação:")
print(f"   COMPRA    : {df_compra.shape}    | ids únicos: {df_compra['id_compra'].nunique()}")
print(f"   ITEM      : {df_item.shape}  | pares únicos: {df_item[['id_compra','id_compra_item']].drop_duplicates().shape[0]}")
print(f"   RESULTADO : {df_resultado.shape}  | pares únicos: {df_resultado[['id_compra','id_compra_item']].drop_duplicates().shape[0]}")
 
# ── 4.3 Validação de unicidade ─────────────────────────────────────────────
assert df_compra["id_compra"].nunique() == len(df_compra), \
    " COMPRA ainda tem id_compra duplicado!"
assert df_item[["id_compra","id_compra_item"]].drop_duplicates().shape[0] == len(df_item), \
    " ITEM ainda tem par duplicado!"
assert df_resultado[["id_compra","id_compra_item"]].drop_duplicates().shape[0] == len(df_resultado), \
    "RESULTADO ainda tem par duplicado!"
print("\n Unicidade validada.")

Após filtro 'Informado': (4323717, 52)

 Após deduplicação:
   COMPRA    : (616588, 72)    | ids únicos: 616588
   ITEM      : (6131062, 62)  | pares únicos: 6131062
   RESULTADO : (4250863, 52)  | pares únicos: 4250863

 Unicidade validada.


5. Merge e Construção da Bigtable

In [5]:
# Colunas de compra a trazer para a bigtable (evita duplicar sufixos)
COLUNAS_COMPRA_MERGE = [
    "id_compra",
    "orgao_entidade_cnpj", "orgao_entidade_razao_social",
    "unidade_orgao_codigo_unidade", "unidade_orgao_uf_sigla",
    "unidade_orgao_municipio_nome", "modalidade_nome",
    "modo_disputa_nome", "data_publicacao_pncp",
    "data_abertura_proposta_pncp", "data_encerramento_proposta_pncp",
    "valor_total_homologado", "numero_controle_PNCP", 
    "ano_compra", "sequencial_compra",
    "objeto_compra",
]
colunas_compra_disponiveis = [c for c in COLUNAS_COMPRA_MERGE if c in df_compra.columns]
 
# Colunas de item a trazer
COLUNAS_ITEM_MERGE = [
    "id_compra", "id_compra_item",
    "numero_item", "descricao", "material_ou_servico",
    "quantidade", "unidade_medida", "valor_unitario_estimado",
    "valor_total", "item_categoria_id", "criterio_julgamento_id",
    "tipo_beneficio", "incentivo_produtivo_basico",
    "orcamento_sigiloso", "tem_resultado",
    "codigo_NCM", "numero_controle_PNCP_compra",
]
colunas_item_disponiveis = [c for c in COLUNAS_ITEM_MERGE if c in df_item.columns]
 
# Merge: RESULTADO ← ITEM ← COMPRA
df_resultado_sub = df_resultado.merge(
    df_item[colunas_item_disponiveis],
    on=["id_compra", "id_compra_item"],
    how="left",
    suffixes=("", "_item"),
)
 
df_base = df_resultado_sub.merge(
    df_compra[colunas_compra_disponiveis],
    on="id_compra",
    how="left",
    suffixes=("", "_compra"),
)
 
# Substituir inf por NaN
df_base.replace([np.inf, -np.inf], np.nan, inplace=True)
 
# Validação pós-merge
assert len(df_base) == len(df_resultado), \
    f"Merge alterou o número de linhas! {len(df_base)} vs {len(df_resultado)}"
 
print(f"Bigtable construída: {df_base.shape}")

C:\Users\gabriel.pessoli_v4co\AppData\Local\Temp\ipykernel_4640\1383213214.py:43: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_base.replace([np.inf, -np.inf], np.nan, inplace=True)


Bigtable construída: (4250863, 82)


6. Feature Engineering

In [6]:
# ── 6.1 Razões e diferenças de preço ──────────────────────────────────────
if {"valor_unitario_homologado", "valor_unitario_estimado"}.issubset(df_base.columns):
    mask_unit = (
        df_base["valor_unitario_estimado"].notna() &
        (df_base["valor_unitario_estimado"] > 0)
    )
    df_base["ratio_preco_unitario"] = np.where(
        mask_unit,
        df_base["valor_unitario_homologado"] / df_base["valor_unitario_estimado"],
        np.nan,
    )
    df_base["diff_preco_unitario"] = (
        df_base["valor_unitario_homologado"] - df_base["valor_unitario_estimado"]
    )
    df_base["diff_preco_unitario_pct"] = np.where(
        mask_unit,
        (df_base["valor_unitario_homologado"] - df_base["valor_unitario_estimado"])
        / df_base["valor_unitario_estimado"],
        np.nan,
    )
 
# ratio_preco_total: valor_total do ITEM (resultado) vs valor_total do ITEM (estimado)
# valor_total no item = quantidade * valor_unitario_estimado
# valor_total_homologado = quantidade_homologada * valor_unitario_homologado

if {"valor_total_homologado", "valor_total"}.issubset(df_base.columns):
    # valor_total = campo da tabela ITEM (quantidade * unitario estimado)
    mask_tot = (
        df_base["valor_total"].notna() &
        (df_base["valor_total"] > 0)
    )
    df_base["ratio_preco_total"] = np.where(
        mask_tot,
        df_base["valor_total_homologado"] / df_base["valor_total"],
        np.nan,
    )
    df_base["diff_preco_total"] = (
        df_base["valor_total_homologado"] - df_base["valor_total"]
    )
else:
    # fallback: usar unitário como proxy do total
    df_base["ratio_preco_total"] = df_base.get("ratio_preco_unitario", np.nan)
 
# Winsorização (mantém igual)
def winsorizar(series: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    q_low  = series.quantile(lower)
    q_high = series.quantile(upper)
    return series.clip(q_low, q_high)
 
for col in ["ratio_preco_unitario", "ratio_preco_total"]:
    if col in df_base.columns:
        df_base[col + "_wins"] = winsorizar(df_base[col])
 
# ── 6.3 Transformações logarítmicas ────────────────────────────────────────
for col in ["valor_unitario_homologado", "valor_total_homologado",
            "valor_unitario_estimado", "valor_total"]:
    if col in df_base.columns:
        df_base[f"log_{col}"] = np.log1p(df_base[col].clip(lower=0))
 
# ── 6.4 Features temporais ─────────────────────────────────────────────────
if "data_publicacao_pncp" in df_base.columns and "data_resultado" in df_base.columns:
    df_base["dias_publicacao_ate_resultado"] = (
        df_base["data_resultado"] - df_base["data_publicacao_pncp"]
    ).dt.days
 
if "data_publicacao_pncp" in df_base.columns and "data_abertura_proposta_pncp" in df_base.columns:
    df_base["dias_publicacao_ate_abertura"] = (
        df_base["data_abertura_proposta_pncp"] - df_base["data_publicacao_pncp"]
    ).dt.days
 
if "data_abertura_proposta_pncp" in df_base.columns and "data_encerramento_proposta_pncp" in df_base.columns:
    df_base["dias_abertura_ate_encerramento"] = (
        df_base["data_encerramento_proposta_pncp"] - df_base["data_abertura_proposta_pncp"]
    ).dt.days
 
if "data_resultado" in df_base.columns:
    df_base["ano_resultado"]        = df_base["data_resultado"].dt.year
    df_base["mes_resultado"]        = df_base["data_resultado"].dt.month
    df_base["dia_semana_resultado"] = df_base["data_resultado"].dt.dayofweek
 
# ── 6.5 Flag publicação retroativa ─────────────────────────────────────────
if {"data_resultado_pncp", "data_publicacao_pncp"}.issubset(df_base.columns):
    df_base["flag_publicacao_retroativa"] = (
        df_base["data_resultado_pncp"] < df_base["data_publicacao_pncp"]
    ).astype(int)
elif {"data_resultado", "data_publicacao_pncp"}.issubset(df_base.columns):
    df_base["flag_publicacao_retroativa"] = (
        df_base["data_resultado"] < df_base["data_publicacao_pncp"]
    ).astype(int)
 
# ── 6.6 Frequência do fornecedor (concentração de mercado) ─────────────────
if "ni_fornecedor" in df_base.columns:
    df_base["freq_fornecedor_abs"] = df_base.groupby("ni_fornecedor")["ni_fornecedor"].transform("count")
 
    if "orgao_entidade_cnpj" in df_base.columns:
        df_base["freq_fornecedor_orgao"] = (
            df_base.groupby(["ni_fornecedor", "orgao_entidade_cnpj"])["ni_fornecedor"]
            .transform("count")
        )
 
    if "unidade_orgao_uf_sigla" in df_base.columns:
        df_base["freq_fornecedor_uf"] = (
            df_base.groupby(["ni_fornecedor", "unidade_orgao_uf_sigla"])["ni_fornecedor"]
            .transform("count")
        )
 
# ── 6.7 Referência de preço por classe NCM ─────────────────────────────────
if {"codigo_NCM", "valor_unitario_homologado"}.issubset(df_base.columns):
    stats_ncm = (
        df_base.groupby("codigo_NCM")["valor_unitario_homologado"]
        .agg(media_valor_ncm="mean", mediana_valor_ncm="median")
        .reset_index()
    )
    df_base = df_base.merge(stats_ncm, on="codigo_NCM", how="left")
 
    mask_ncm = df_base["media_valor_ncm"].notna() & (df_base["media_valor_ncm"] > 0)
    df_base["ratio_vs_media_ncm"] = np.where(
        mask_ncm,
        df_base["valor_unitario_homologado"] / df_base["media_valor_ncm"],
        np.nan,
    )
 
# ── 6.8 Quantidade de itens por compra ─────────────────────────────────────
if {"id_compra", "id_compra_item"}.issubset(df_base.columns):
    df_base["qtd_itens_compra"] = (
        df_base.groupby("id_compra")["id_compra_item"].transform("nunique")
    )
 
# ── 6.9 Flags de qualidade de dados ────────────────────────────────────────
if "valor_unitario_estimado" in df_base.columns:
    df_base["flag_valor_unitario_estimado_zero_ou_nulo"] = (
        (df_base["valor_unitario_estimado"].isna()) |
        (df_base["valor_unitario_estimado"] == 0)
    ).astype(int)
 
if "valor_total_homologado" in df_base.columns:
    df_base["flag_valor_total_homologado_zero_ou_nulo"] = (
        (df_base["valor_total_homologado"].isna()) |
        (df_base["valor_total_homologado"] == 0)
    ).astype(int)
 
if "unidade_orgao_uf_sigla" in df_base.columns:
    df_base["flag_uf_ausente"] = df_base["unidade_orgao_uf_sigla"].isna().astype(int)
 
print(f"Feature engineering concluído. Shape: {df_base.shape}")

Feature engineering concluído. Shape: (4250863, 110)


7. Limpeza Final e Salvamento

In [7]:
# ── 7.1 Remover colunas com >90% de nulos ──────────────────────────────────
thresh_nulo = 0.90
pct_nulos   = df_base.isnull().mean()
colunas_excluir = pct_nulos[pct_nulos > thresh_nulo].index.tolist()
 
print(f"Colunas com >{thresh_nulo*100:.0f}% de nulos removidas ({len(colunas_excluir)}):")
print(colunas_excluir)
 
df_modelo = df_base.drop(columns=colunas_excluir).copy()
 
# ── 7.2 Resumo pós-limpeza ─────────────────────────────────────────────────
print(f"\n✅ Shape após limpeza: {df_modelo.shape}")
print(f"\nEstatísticas das principais variáveis numéricas:")
cols_stats = [c for c in [
    "ratio_preco_unitario", "ratio_preco_total",
    "valor_unitario_homologado", "valor_total_homologado",
    "dias_publicacao_ate_resultado",
] if c in df_modelo.columns]
print(df_modelo[cols_stats].describe())
 
# ── 7.3 Salvar em Parquet (evita DtypeWarning na releitura e é ~5x menor) ──
try:
    df_modelo.to_parquet(ARQUIVO_FINAL, index=False)
    print(f"\n Base salva em: {ARQUIVO_FINAL}")
except Exception as e:
    csv_fallback = ARQUIVO_FINAL.with_suffix(".csv")
    df_modelo.to_csv(csv_fallback, index=False)
    print(f" Parquet falhou ({e}). Salvo em CSV: {csv_fallback}")

Colunas com >90% de nulos removidas (18):
['data_cancelamento', 'motivo_cancelamento', 'amparo_legal_margem_preferencia_id', 'amparo_legal_margem_preferencia_nome', 'amparo_legal_margem_preferencia_descricao', 'amparo_legal_criterio_desempate_id', 'amparo_legal_criterio_desempate_nome', 'amparo_legal_criterio_desempate_descricao', 'moeda_estrangeira_id', 'data_cotacao_moeda_estrangeira', 'valor_nominal_moeda_estrangeira', 'pais_origem_produto_servico_id', 'timezone_cotacao_moeda_estrangeira', 'data_cancelamento_pncp', 'codigo_NCM', 'media_valor_ncm', 'mediana_valor_ncm', 'ratio_vs_media_ncm']

✅ Shape após limpeza: (4250863, 92)

Estatísticas das principais variáveis numéricas:
       ratio_preco_unitario  ratio_preco_total  valor_unitario_homologado  \
count          4.213563e+06       4.213068e+06               4.250863e+06   
mean           4.701620e+04       5.785044e+04               7.824313e+04   
std            4.024787e+07       4.211971e+07               9.859114e+06   
min  